# 04 – Geo-aware label smoothing (trên đặc trưng XLS-R đã cache)

**Bài học từ notebook 03:**
- SORD với τ ≥ 50 km chỉ để lại khoảng 40% xác suất mục tiêu cho đúng tỉnh, vì các tỉnh láng giềng chỉ cách nhau trung vị 44 km. Kết quả là accuracy giảm 7,7 điểm.
- Label smoothing (LS) đồng đều tăng accuracy thêm 3,4 điểm, nhưng làm khoảng cách sai xấu đi.

**Đề xuất: Geo-LS.** Giữ `1−ε` xác suất cho đúng tỉnh, và chia phần `ε` còn lại cho các tỉnh **khác** theo `exp(−d/τ)` thay vì chia đều.
Mục tiêu là giữ được lợi ích điều chuẩn của LS, đồng thời làm lỗi rơi về tỉnh gần hơn.

Lưới thử nghiệm (mỗi cấu hình 3 seed, chọn tham số theo valid):
- `CE`
- `LS ε`
- `SORD τ` nhỏ (5–30 km)
- `Geo-LS (ε, τ)`

Notebook cũng kiểm tra thêm **dấu vân tay kênh thu theo từng tỉnh**: accuracy LTAS của mỗi tỉnh so với accuracy XLS-R.

- **Runtime:** GPU T4/L4, khoảng 20–25 phút.
- **Kết quả:** `results/geo_ls_report.json` và `fig_pareto.png`. Gửi cả hai cho Claude.

In [ ]:
# ==== CẤU HÌNH ====
DRIVE_ROOT = "/content/drive/MyDrive/vimd_dl"
PROVINCE_COL = "province_code"
SEEDS = [0, 1, 2]
LS_EPS = [0.1, 0.2, 0.3]
SORD_TAUS = [5, 10, 20, 30]
GEOLS_EPS = [0.1, 0.2, 0.3]
GEOLS_TAUS = [25, 50, 100, 200]

In [ ]:
import os, glob, json, time, warnings
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.metrics import f1_score, top_k_accuracy_score
from scipy.stats import spearmanr
warnings.filterwarnings("ignore")
T0 = time.time()
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass
FEAT_DIR, RES_DIR = f"{DRIVE_ROOT}/features", f"{DRIVE_ROOT}/results"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Thiết bị:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "")

meta = pd.read_csv(f"{DRIVE_ROOT}/metadata.csv", low_memory=False)
meta[PROVINCE_COL] = meta[PROVINCE_COL].astype(str)
prov = pd.read_csv(f"{DRIVE_ROOT}/provinces.csv"); prov[PROVINCE_COL] = prov[PROVINCE_COL].astype(str)
CLASSES = sorted(meta[PROVINCE_COL].unique()); C = len(CLASSES)
y = meta[PROVINCE_COL].map({c: i for i, c in enumerate(CLASSES)}).values
use = ~meta["exclude"].astype(bool).values
tr, va, te = [(meta["split"].values == s) & use for s in ("train", "valid", "test")]
pidx = prov.set_index(PROVINCE_COL)
NAMES = [pidx.loc[c, "province_name"] for c in CLASSES]
region_of = np.array([pidx.loc[c, "region"] for c in CLASSES])
yt = y[te]

# Tọa độ tỉnh lỵ (giống notebook 03)
COORD = {
 "CaoBang": (22.666, 106.258), "LangSon": (21.853, 106.761), "QuangNinh": (20.951, 107.080), "HaiPhong": (20.845, 106.688),
 "ThaiBinh": (20.446, 106.336), "NamDinh": (20.420, 106.168), "PhuTho": (21.322, 105.402), "ThaiNguyen": (21.594, 105.848),
 "YenBai": (21.705, 104.875), "TuyenQuang": (21.823, 105.214), "HaGiang": (22.823, 104.984), "LaoCai": (22.486, 103.971),
 "LaiChau": (22.396, 103.458), "SonLa": (21.326, 103.917), "DienBien": (21.386, 103.023), "HoaBinh": (20.817, 105.338),
 "HaNoi": (21.028, 105.854), "HaiDuong": (20.940, 106.333), "NinhBinh": (20.251, 105.975), "VinhPhuc": (21.310, 105.597),
 "HungYen": (20.646, 106.051), "HaNam": (20.541, 105.914), "BacKan": (22.147, 105.835), "BacGiang": (21.273, 106.195),
 "BacNinh": (21.186, 106.076), "ThanhHoa": (19.807, 105.776), "NgheAn": (18.679, 105.682), "HaTinh": (18.343, 105.906),
 "QuangBinh": (17.468, 106.622), "QuangTri": (16.816, 107.100), "ThuaThienHue": (16.464, 107.586), "DaNang": (16.054, 108.202),
 "QuangNam": (15.573, 108.474), "QuangNgai": (15.120, 108.792), "BinhDinh": (13.776, 109.224), "PhuYen": (13.096, 109.309),
 "KhanhHoa": (12.239, 109.197), "NinhThuan": (11.565, 108.988), "BinhThuan": (10.928, 108.102), "KonTum": (14.350, 108.000),
 "GiaLai": (13.983, 108.000), "DakLak": (12.667, 108.038), "DakNong": (12.004, 107.690), "LamDong": (11.940, 108.458),
 "HoChiMinh": (10.776, 106.701), "DongNai": (10.957, 106.843), "BinhDuong": (10.980, 106.651), "BaRiaVungTau": (10.496, 107.169),
 "TayNinh": (11.310, 106.098), "BinhPhuoc": (11.535, 106.883), "LongAn": (10.535, 106.413), "TienGiang": (10.360, 106.360),
 "BenTre": (10.241, 106.376), "VinhLong": (10.254, 105.972), "TraVinh": (9.935, 106.345), "CanTho": (10.045, 105.747),
 "HauGiang": (9.784, 105.470), "SocTrang": (9.603, 105.980), "BacLieu": (9.294, 105.727), "CaMau": (9.176, 105.150),
 "DongThap": (10.460, 105.633), "AnGiang": (10.386, 105.435), "KienGiang": (10.012, 105.081),
}
LAT = np.array([COORD[n][0] for n in NAMES]); LON = np.array([COORD[n][1] for n in NAMES])
p_ = np.pi / 180
DIST = 2 * 6371 * np.arcsin(np.sqrt(np.sin((LAT[None] - LAT[:, None]) * p_ / 2) ** 2 + np.cos(LAT[:, None] * p_)
                                    * np.cos(LAT[None] * p_) * np.sin((LON[None] - LON[:, None]) * p_ / 2) ** 2))
print(f"{C} tỉnh | train {tr.sum()} | valid {va.sum()} | test {te.sum()}")

## 1. Dấu vân tay kênh thu theo tỉnh

In [ ]:
pred_ltas = np.load(f"{RES_DIR}/pred_acoustic_ltas.npy")
pred_xlsr = np.load(f"{RES_DIR}/pred_xlsr_wsum_mlp.npy")
acc_by = lambda p: np.array([(p[yt == k] == k).mean() for k in range(C)])
A_ltas, A_xlsr = acc_by(pred_ltas), acc_by(pred_xlsr)
rho_ch, p_ch = spearmanr(A_ltas, A_xlsr)
chan = pd.DataFrame(dict(province=NAMES, ltas_acc=A_ltas.round(3), xlsr_acc=A_xlsr.round(3))).sort_values("ltas_acc", ascending=False)
CHANNEL = dict(spearman_ltas_vs_xlsr_per_province=round(float(rho_ch), 3), p_value=float(p_ch),
               top_ltas_provinces=chan.head(10).to_dict(orient="records"))
print(f"Spearman(LTAS acc, XLS-R acc) theo tỉnh = {rho_ch:.2f} (p={p_ch:.1e})")
print(chan.head(10).to_string(index=False))

## 2. Lưới hàm mục tiêu trên head weighted-sum + MLP

In [ ]:
fdir = sorted(glob.glob(f"{FEAT_DIR}/xlsr_*"))[0]
GROUPS = {s: g.index.values for s, g in meta.groupby("shard")}
first = np.load(f"{fdir}/{next(iter(GROUPS))}.npy")
XL = np.zeros((len(meta),) + first.shape[1:], np.float16)
for s, idx in GROUPS.items():
    XL[idx] = np.load(f"{fdir}/{s}.npy")
Xg = torch.tensor(XL, device=device); del XL
mtr = torch.tensor(tr, device=device)
mu, sd = Xg[mtr].float().mean(0), Xg[mtr].float().std(0) + 1e-5
for i in range(0, len(Xg), 2048):
    Xg[i:i + 2048] = ((Xg[i:i + 2048].float() - mu) / sd).half()
Xtr_, Xva_, Xte_ = [Xg[torch.tensor(m, device=device)] for m in (tr, va, te)]
del Xg
nL, D = Xtr_.shape[1], Xtr_.shape[2]
ytr_ = torch.tensor(y[tr], device=device)
DIST_t = torch.tensor(DIST, dtype=torch.float32, device=device)
EYE = torch.eye(C, device=device)

def target_matrix(kind, p):
    """Trả về ma trận (C, C): hàng i là phân phối mục tiêu khi nhãn thật là i."""
    if kind == "ce":
        return EYE
    if kind == "ls":
        return (1 - p) * EYE + p / C
    if kind == "sord":
        return torch.softmax(-DIST_t / p, 1)
    if kind == "geols":
        eps, tau = p
        nb = torch.exp(-DIST_t / tau) * (1 - EYE)
        return (1 - eps) * EYE + eps * nb / nb.sum(1, keepdim=True)
    raise ValueError(kind)

class Head(nn.Module):
    def __init__(s):
        super().__init__(); s.w = nn.Parameter(torch.zeros(nL))
        s.mlp = nn.Sequential(nn.Linear(D, 512), nn.GELU(), nn.Dropout(0.3), nn.Linear(512, C))
    def forward(s, x):
        return s.mlp((x.float() * s.w.softmax(0)[None, :, None]).sum(1))

@torch.no_grad()
def predict(h, X, bs=2048):
    h.eval()
    return torch.cat([h(X[i:i + bs]).softmax(-1) for i in range(0, len(X), bs)]).cpu().numpy()

def train(kind, p, seed, epochs=80, lr=1e-3, wd=1e-3, bs=256, patience=10):
    torch.manual_seed(seed)
    h = Head().to(device); opt = torch.optim.AdamW(h.parameters(), lr=lr, weight_decay=wd)
    Q = target_matrix(kind, p); best, state, bad = -1, None, 0
    for ep in range(epochs):
        h.train(); perm = torch.randperm(len(Xtr_), device=device)
        for i in range(0, len(perm), bs):
            b = perm[i:i + bs]
            loss = -(Q[ytr_[b]] * F.log_softmax(h(Xtr_[b]), -1)).sum(1).mean()
            opt.zero_grad(); loss.backward(); opt.step()
        acc = (predict(h, Xva_).argmax(1) == y[va]).mean()
        if acc > best:
            best, state, bad = acc, {k: v.clone() for k, v in h.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= patience:
                break
    h.load_state_dict(state)
    return h

def metrics(P, yy):
    pred = P.argmax(1); d = DIST[yy, pred]
    return dict(acc=(pred == yy).mean(), macro_f1=f1_score(yy, pred, average="macro"),
                top5=top_k_accuracy_score(yy, P, k=5, labels=np.arange(C)),
                region_acc=(region_of[pred] == region_of[yy]).mean(), mean_err_km=d.mean(),
                within_100km=(d <= 100).mean(), expected_km=(P * DIST[yy]).sum(1).mean(),
                nll=-np.log(P[np.arange(len(yy)), yy] + 1e-12).mean())

GRID = [("ce", None)] + [("ls", e) for e in LS_EPS] + [("sord", t) for t in SORD_TAUS] \
       + [("geols", (e, t)) for e in GEOLS_EPS for t in GEOLS_TAUS]
RUNS, PROBS = [], {}
for kind, p in GRID:
    t = time.time()
    for seed in SEEDS:
        h = train(kind, p, seed)
        mv, Pt = metrics(predict(h, Xva_), y[va]), predict(h, Xte_)
        mt = metrics(Pt, yt); PROBS[(kind, str(p), seed)] = Pt
        RUNS.append(dict(kind=kind, param=str(p), seed=seed, **{f"valid_{k}": float(v) for k, v in mv.items()},
                         **{f"test_{k}": float(v) for k, v in mt.items()}))
    s = pd.DataFrame(RUNS[-len(SEEDS):])
    print(f"{kind:6s} {str(p):12s} | valid {s.valid_acc.mean():.4f} | test {s.test_acc.mean():.4f}±{s.test_acc.std():.4f}"
          f" | err {s.test_mean_err_km.mean():.0f} km | {time.time() - t:.0f}s", flush=True)
runs = pd.DataFrame(RUNS)

## 3. Chọn tham số theo valid, so sánh cặp với CE và LS

In [ ]:
mv = runs.groupby(["kind", "param"])["valid_acc"].mean().reset_index()
chosen = mv.loc[mv.groupby("kind")["valid_acc"].idxmax()][["kind", "param"]].values.tolist()
cols = ["acc", "macro_f1", "top5", "region_acc", "mean_err_km", "within_100km", "expected_km", "nll"]
MAIN = []
for k, p in chosen:
    s = runs[(runs.kind == k) & (runs.param == p)]
    MAIN.append(dict(kind=k, param=p, **{c: f"{s['test_' + c].mean():.4f} ± {s['test_' + c].std():.4f}" for c in cols}))
MAIN = pd.DataFrame(MAIN)
print(MAIN.to_string(index=False))

ens = lambda k, p: np.mean([PROBS[(k, p, s)] for s in SEEDS], 0)
B = np.random.default_rng(0).integers(0, len(yt), (2000, len(yt)))
def paired(Pa, Pb):
    a, b = Pa.argmax(1), Pb.argmax(1)
    dacc = (a == yt).astype(float) - (b == yt); dkm = DIST[yt, a] - DIST[yt, b]
    return dict(acc_diff=round(float(dacc.mean()), 4), acc_ci95=np.percentile(dacc[B].mean(1), [2.5, 97.5]).round(4).tolist(),
                km_diff=round(float(dkm.mean()), 1), km_ci95=np.percentile(dkm[B].mean(1), [2.5, 97.5]).round(1).tolist())
best = dict(chosen)
COMPARE = {}
for k, p in chosen:
    for ref in ("ce", "ls"):
        if k != ref and ref in best:
            COMPARE[f"{k}({p}) - {ref}({best[ref]})"] = paired(ens(k, p), ens(ref, best[ref]))
print(json.dumps(COMPARE, indent=1))

## 4. Hình: đánh đổi accuracy và khoảng cách sai (mỗi điểm là một cấu hình, trung bình 3 seed)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
plt.rcParams.update({"figure.dpi": 130, "axes.spines.top": False, "axes.spines.right": False, "font.size": 9,
                     "axes.facecolor": "#fcfcfb", "figure.facecolor": "#fcfcfb", "axes.edgecolor": "#52514e",
                     "xtick.color": "#52514e", "ytick.color": "#52514e"})
STY = {"ce": ("#2a78d6", "o", "CE"), "ls": ("#eb6834", "s", "LS"), "sord": ("#1baf7a", "^", "SORD"),
       "geols": ("#eda100", "D", "Geo-LS")}
agg = runs.groupby(["kind", "param"])[["test_acc", "test_mean_err_km"]].mean().reset_index()
fig, ax = plt.subplots(figsize=(5.4, 3.6))
for k, (col, mk, lab) in STY.items():
    a = agg[agg.kind == k]
    ax.scatter(a.test_mean_err_km, a.test_acc, c=col, marker=mk, s=46, edgecolors="#fcfcfb", linewidths=1.5, zorder=3)
for k, p in chosen:
    r = agg[(agg.kind == k) & (agg.param == p)].iloc[0]
    ax.annotate(f"{STY[k][2]} {p if p != 'None' else ''}", (r.test_mean_err_km, r.test_acc), fontsize=7,
                color="#52514e", xytext=(5, 3), textcoords="offset points")
ax.set_xlabel("Khoảng cách sai trung bình (km) ← tốt hơn"); ax.set_ylabel("Accuracy test → tốt hơn")
ax.set_title("Đánh đổi accuracy – khoảng cách sai (đặc trưng XLS-R đóng băng)", fontsize=9, loc="left")
ax.grid(color="#e6e5e0", linewidth=0.6); ax.set_axisbelow(True)
ax.legend(handles=[Line2D([], [], ls="", marker=m, color=c, markersize=7, label=l) for c, m, l in STY.values()],
          frameon=False, loc="lower left")
fig.tight_layout(); fig.savefig(f"{RES_DIR}/fig_pareto.png"); plt.show()

report = dict(channel=CHANNEL, main=MAIN.to_dict(orient="records"), compare=COMPARE,
              chosen={k: p for k, p in chosen}, all_runs=runs.round(4).to_dict(orient="records"),
              gpu=torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu",
              total_minutes=round((time.time() - T0) / 60, 1))
json.dump(report, open(f"{RES_DIR}/geo_ls_report.json", "w"), ensure_ascii=False, indent=1, default=str)
print("Đã lưu results/geo_ls_report.json + fig_pareto.png – gửi cho Claude.")